# Exercise XP — Agentic AI with RAG

## Complete and thoroughly commented solution

This notebook builds a small agent-like Retrieval-Augmented Generation
workflow using open-source Python packages only.

The system contains:

- an in-memory knowledge base;
- FAISS vector retrieval with fake embeddings;
- a free Wikipedia fallback;
- a transparent rule-based planner;
- an evidence-aware answer function;
- inline source citations;
- graceful handling of weak or missing evidence;
- a stub chat model by default;
- an optional tiny Hugging Face generator.

No API key is required.

## What makes the workflow agent-like?

The application does more than send one prompt to a model.

```text
Question
   ↓
Rule-based planner
   ├── known internal topic → search the KB
   └── unknown topic → search Wikipedia
                          ↓
                 inspect evidence strength
                          ↓
               optional Wikipedia fallback
                          ↓
               grounded answer + citations
```

The planner chooses an information source, the tools collect evidence, and
the answer function checks whether that evidence is sufficient before
producing a response.

## Important limitation

`FakeEmbeddings` produces artificial vectors. It is useful for learning the
LangChain and FAISS interfaces, but it does not represent real semantic
meaning as reliably as a trained embedding model.

For that reason, this notebook also:

- keeps all internal documents within one related domain;
- uses a rule-based planner;
- calculates simple lexical overlap;
- explicitly reports thin evidence;
- can fall back to Wikipedia.

# 0. Environment setup

In [ ]:
# Install all packages requested by the exercise.
#
# Version ranges keep the notebook within the current LangChain major
# version while still allowing compatible bug fixes.

%pip install -qU \
    "langchain>=1.0,<2.0" \
    "langchain-core>=1.0,<2.0" \
    "langchain-community>=0.4,<0.5" \
    "faiss-cpu>=1.8" \
    "wikipedia>=1.4" \
    "transformers>=4.45,<5.0" \
    "accelerate>=1.0,<2.0" \
    "sentencepiece>=0.2,<1.0"

In a fresh Colab runtime, continue normally after installation. If Colab
reports that an already-imported package must be restarted, restart the
runtime once and run the notebook again from the top.

In [ ]:
# Standard-library utilities.
import importlib.metadata as metadata
import re
import warnings
from functools import lru_cache
from typing import Any

# Data display.
import pandas as pd

# LangChain document, prompt, retrieval, and testing components.
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import FakeEmbeddings
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS

# Fake chat models have moved between LangChain packages over time.
# Prefer the modern langchain-core location and keep a compatibility
# fallback for environments that still expose the community path.
try:
    from langchain_core.language_models.fake_chat_models import (
        FakeListChatModel,
    )
except ImportError:
    from langchain_community.chat_models.fake import FakeListChatModel

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Installed versions:")

for package_name in [
    "langchain",
    "langchain-core",
    "langchain-community",
    "faiss-cpu",
    "wikipedia",
    "transformers",
]:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: not found")

# Exercise 1 — Build the KB retriever

## Knowledge-base design

Each `Document` contains:

- `page_content`: the text that can be retrieved;
- `metadata["source"]`: the identifier used in citations;
- `metadata["title"]`: a readable label;
- `metadata["topics"]`: keywords used by the planner.

The source identifiers already include the `kb:` prefix, so a final
citation can be written directly as `[kb:retrievers]`.

In [ ]:
# Seven short documents form the internal knowledge base.
#
# They intentionally cover related topics so the FAISS exercise remains
# useful even though FakeEmbeddings are not semantically trained.

kb_docs = [
    Document(
        page_content=(
            "Agentic systems interpret a goal, inspect available context, "
            "choose whether a tool is needed, execute a short tool plan, "
            "and then synthesize an answer from the observed results. "
            "They should avoid unnecessary tool calls and reconsider the "
            "plan after every tool result."
        ),
        metadata={
            "source": "kb:agentic_loop",
            "title": "Agentic decision loop",
            "topics": [
                "agentic",
                "agent",
                "tool",
                "planner",
                "planning",
            ],
        },
    ),
    Document(
        page_content=(
            "A retriever searches an internal knowledge base for passages "
            "related to a user query. A focused top-k value keeps context "
            "small. If results are noisy, the query should be reformulated "
            "or filtered before the system answers."
        ),
        metadata={
            "source": "kb:retrievers",
            "title": "Retriever fundamentals",
            "topics": [
                "retriever",
                "retrieval",
                "vector",
                "faiss",
                "top-k",
            ],
        },
    ),
    Document(
        page_content=(
            "Retrieval-Augmented Generation, or RAG, combines retrieved "
            "external evidence with a language model response. Retrieval "
            "does not guarantee correctness: the system must inspect the "
            "passages and avoid claims not supported by them."
        ),
        metadata={
            "source": "kb:rag",
            "title": "RAG fundamentals",
            "topics": [
                "rag",
                "retrieval-augmented",
                "grounding",
                "context",
            ],
        },
    ),
    Document(
        page_content=(
            "Grounded answers place citations close to the claims they "
            "support. A citation should identify the actual retrieved "
            "document. The system must never invent document names, URLs, "
            "or source identifiers."
        ),
        metadata={
            "source": "kb:citations",
            "title": "Grounding and citation rules",
            "topics": [
                "citation",
                "cite",
                "source",
                "ground",
                "grounded",
            ],
        },
    ),
    Document(
        page_content=(
            "When evidence is missing, weak, ambiguous, or conflicting, "
            "the system should state that limitation. It should separate "
            "supported facts from inference and suggest a clearer or more "
            "specific follow-up query instead of fabricating details."
        ),
        metadata={
            "source": "kb:honesty",
            "title": "Evidence-aware honesty",
            "topics": [
                "honest",
                "honesty",
                "transparent",
                "uncertain",
                "evidence",
                "missing",
            ],
        },
    ),
    Document(
        page_content=(
            "Wikipedia is a free broad-coverage fallback when a curated "
            "knowledge base does not cover a topic. It is suitable for "
            "general background and definitions, but sensitive claims "
            "should be checked against more authoritative sources."
        ),
        metadata={
            "source": "kb:wikipedia_policy",
            "title": "When to use Wikipedia",
            "topics": [
                "wikipedia",
                "external",
                "fallback",
                "background",
            ],
        },
    ),
    Document(
        page_content=(
            "A concise grounded answer should lead with the main finding, "
            "then provide one or two supporting statements with inline "
            "citations. Longer explanations are useful only when the user "
            "asks for detail or when safety and uncertainty require it."
        ),
        metadata={
            "source": "kb:answer_style",
            "title": "Answer style",
            "topics": [
                "answer",
                "concise",
                "style",
                "explain",
            ],
        },
    ),
]

print("Knowledge-base documents:", len(kb_docs))

display(pd.DataFrame([
    {
        "source": document.metadata["source"],
        "title": document.metadata["title"],
        "topics": ", ".join(document.metadata["topics"]),
    }
    for document in kb_docs
]))

In [ ]:
# FakeEmbeddings satisfies the exercise without downloading a real model.
#
# The vector size only needs to be consistent between stored documents and
# queries. A larger size does not make fake vectors semantically smarter.

embeddings = FakeEmbeddings(size=256)

# FAISS.from_documents:
# 1. embeds every Document;
# 2. stores vectors in a FAISS index;
# 3. keeps the original Documents in an associated document store.
vector_store = FAISS.from_documents(
    documents=kb_docs,
    embedding=embeddings,
)

# The retriever interface gives the rest of the application a simple
# `.invoke(question)` method and returns exactly three Documents.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("FAISS knowledge base ready.")

## Inspect one retrieval

Because fake embeddings are artificial, the exact document order can vary.
The objective of this cell is to confirm that the retriever returns three
LangChain `Document` objects carrying their source metadata.

In [ ]:
sample_retrieval = retriever.invoke(
    "How should a system cite retrieved evidence?"
)

print("Retrieved documents:", len(sample_retrieval))

for rank, document in enumerate(sample_retrieval, start=1):
    print(
        f"{rank}. {document.metadata['source']} — "
        f"{document.metadata['title']}"
    )

# Exercise 2 — Add the free Wikipedia tool

`WikipediaAPIWrapper` uses the public Wikipedia service through the
`wikipedia` Python package. It does not require an API key.

The helper below returns:

- a list of normalized snippets;
- `None` when no error occurred;
- a readable error message when Wikipedia is unavailable or ambiguous.

In [ ]:
def wikipedia_source_id(title: str) -> str:
    """Convert a Wikipedia title into a stable citation identifier."""

    # Replace whitespace with underscores, matching normal Wikipedia URLs.
    slug = re.sub(r"\s+", "_", title.strip())

    # Keep letters, numbers, underscores, hyphens, and parentheses.
    slug = re.sub(r"[^A-Za-z0-9_()\-]", "", slug)

    return slug or "Unknown_article"


@lru_cache(maxsize=32)
def wiki_search(
    query: str,
    k: int = 2,
) -> tuple[list[dict[str, str]], str | None]:
    """Search Wikipedia and return short, citation-ready snippets."""

    cleaned_query = query.strip()

    if not cleaned_query:
        return [], "Wikipedia query was empty."

    if not isinstance(k, int) or not 1 <= k <= 5:
        return [], "k must be an integer from 1 to 5."

    try:
        # A new wrapper is created so each call can respect its own k.
        wrapper = WikipediaAPIWrapper(
            lang="en",
            top_k_results=k,
            doc_content_chars_max=1200,
        )

        # `.load()` returns LangChain Documents with content and metadata.
        documents = wrapper.load(cleaned_query)

        snippets = []

        for document in documents[:k]:
            title = str(
                document.metadata.get("title")
                or "Untitled Wikipedia article"
            ).strip()

            source_url = str(
                document.metadata.get("source")
                or (
                    "https://en.wikipedia.org/wiki/"
                    + wikipedia_source_id(title)
                )
            )

            summary = re.sub(
                r"\s+",
                " ",
                document.page_content,
            ).strip()

            snippets.append({
                "title": title,
                "source_id": (
                    "wiki:"
                    + wikipedia_source_id(title)
                ),
                "url": source_url,
                "summary": summary[:1000],
            })

        if not snippets:
            return [], "Wikipedia returned no matching articles."

        return snippets, None

    except Exception as error:
        # Network failures, ambiguous page titles, and package-level
        # exceptions are converted into evidence metadata instead of
        # crashing the whole agent.
        return [], (
            f"Wikipedia lookup failed: "
            f"{type(error).__name__}: {error}"
        )

In [ ]:
# This cell performs a live network request when Wikipedia is reachable.
# A failure is displayed but does not break later notebook cells.

wiki_demo, wiki_demo_error = wiki_search(
    "Python programming language",
    k=2,
)

if wiki_demo:
    display(pd.DataFrame(wiki_demo))
else:
    print(wiki_demo_error)

# Exercise 3 — Build the rule-based planner

The planner does not use an LLM. It searches the question for topics known
by the internal KB.

Its plan contains:

- `action`: `kb` or `wiki`;
- `reason`: a human-readable explanation;
- `matched_topics`: the matching internal keywords;
- `wiki_fallback_if_thin`: whether weak KB evidence may trigger Wikipedia.

In [ ]:
# Build one normalized set of topics from Document metadata.
KNOWN_KB_TOPICS = sorted({
    topic.lower()
    for document in kb_docs
    for topic in document.metadata.get("topics", [])
})


def tokenize(text: str) -> set[str]:
    """Return lowercase words while removing punctuation."""
    return set(
        re.findall(
            r"[a-z0-9]+(?:-[a-z0-9]+)?",
            text.lower(),
        )
    )


def topic_matches(
    question: str,
) -> list[str]:
    """Find KB topics mentioned by the question."""
    question_lower = question.lower()
    question_tokens = tokenize(question)

    matches = []

    for topic in KNOWN_KB_TOPICS:
        # Multi-word and hyphenated topics are matched as substrings.
        if " " in topic or "-" in topic:
            matched = topic in question_lower
        else:
            matched = topic in question_tokens

        if matched:
            matches.append(topic)

    return matches


def plan(question: str) -> dict[str, Any]:
    """Choose the internal KB when its known topics match."""
    cleaned_question = question.strip()

    if not cleaned_question:
        return {
            "action": "none",
            "reason": "The question is empty.",
            "matched_topics": [],
            "wiki_fallback_if_thin": False,
        }

    matches = topic_matches(cleaned_question)

    if matches:
        return {
            "action": "kb",
            "reason": (
                "The question mentions topics covered by "
                "the internal knowledge base."
            ),
            "matched_topics": matches,
            "wiki_fallback_if_thin": True,
        }

    return {
        "action": "wiki",
        "reason": (
            "No known internal topic matched, so the planner "
            "uses the broad external Wikipedia fallback."
        ),
        "matched_topics": [],
        "wiki_fallback_if_thin": False,
    }

In [ ]:
planner_examples = [
    "How should an agent ground answers?",
    "Who created Python?",
    "How should uncertain evidence be handled?",
]

display(pd.DataFrame([
    {
        "question": question,
        **plan(question),
    }
    for question in planner_examples
]))

# Exercise 4 — Build the evidence-aware answer function

## Evidence-quality check

Fake vector similarity alone is not enough to claim strong relevance.
The notebook calculates lexical overlap between the question and each
retrieved passage.

The overlap score is not a full semantic evaluator. It is a small,
transparent safeguard for this beginner exercise.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can",
    "did", "do", "does", "for", "from", "how", "i", "in", "is",
    "it", "of", "on", "or", "should", "the", "their", "this",
    "to", "use", "was", "what", "when", "where", "which", "who",
    "why", "with",
}


def content_terms(text: str) -> set[str]:
    """Return meaningful lowercase words for simple relevance scoring."""
    return {
        token
        for token in tokenize(text)
        if token not in STOPWORDS and len(token) > 2
    }


def lexical_overlap(
    question: str,
    evidence_text: str,
) -> int:
    """Count meaningful words shared by the question and evidence."""
    return len(
        content_terms(question)
        & content_terms(evidence_text)
    )


def rerank_retrieved_documents(
    question: str,
    documents: list[Document],
) -> list[Document]:
    """Put the most lexically related retrieved passage first."""
    return sorted(
        documents,
        key=lambda document: lexical_overlap(
            question,
            document.page_content,
        ),
        reverse=True,
    )


def kb_evidence_is_thin(
    question: str,
    documents: list[Document],
) -> bool:
    """Identify missing or weak internal evidence."""
    if not documents:
        return True

    best_score = max(
        lexical_overlap(
            question,
            document.page_content,
        )
        for document in documents
    )

    return best_score == 0

## Extractive evidence selection

The default answer does not invent a new fact. It selects the sentence from
each source that overlaps most with the question, then attaches the source
identifier.

This is intentionally simple but grounded.

In [ ]:
def split_sentences(text: str) -> list[str]:
    """Split short English prose into sentence-like segments."""
    return [
        sentence.strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+",
            re.sub(r"\s+", " ", text).strip(),
        )
        if sentence.strip()
    ]


def best_supported_sentence(
    question: str,
    text: str,
) -> str:
    """Select the sentence with the highest lexical overlap."""
    sentences = split_sentences(text)

    if not sentences:
        return ""

    return max(
        sentences,
        key=lambda sentence: (
            lexical_overlap(question, sentence),
            -len(sentence),
        ),
    )


def build_grounded_draft(
    question: str,
    kb_documents: list[Document],
    wiki_snippets: list[dict[str, str]],
    evidence_thin: bool,
    wiki_error: str | None,
) -> str:
    """Create a concise answer containing only source-backed statements."""

    evidence_items = []

    for document in kb_documents:
        evidence_items.append({
            "source_id": document.metadata["source"],
            "text": document.page_content,
        })

    for snippet in wiki_snippets:
        evidence_items.append({
            "source_id": snippet["source_id"],
            "text": snippet["summary"],
        })

    if not evidence_items:
        message = (
            "The available sources do not provide enough evidence "
            "to answer this question reliably."
        )

        if wiki_error:
            message += f" External lookup detail: {wiki_error}"

        return (
            message
            + " Try a more specific follow-up query or provide a "
            "relevant internal document."
        )

    ranked = sorted(
        evidence_items,
        key=lambda item: lexical_overlap(
            question,
            item["text"],
        ),
        reverse=True,
    )

    statements = []

    for item in ranked:
        sentence = best_supported_sentence(
            question,
            item["text"],
        )

        if not sentence:
            continue

        cited_statement = (
            f"{sentence} [{item['source_id']}]"
        )

        if cited_statement not in statements:
            statements.append(cited_statement)

        if len(statements) == 2:
            break

    if not statements:
        return (
            "The retrieved sources were available, but they did not "
            "contain a clear statement supporting an answer. Try a "
            "more specific follow-up query."
        )

    answer = " ".join(statements)

    if evidence_thin:
        answer += (
            " The evidence is limited, so a narrower follow-up query "
            "or an additional authoritative source would improve confidence."
        )

    return answer

## Stub model and optional tiny Hugging Face model

`FakeListChatModel` is the default. It makes the notebook deterministic and
free. The grounded draft is supplied as its predefined response.

The optional Hugging Face branch downloads `sshleifer/tiny-gpt2`. That model
is extremely small and is included only to demonstrate a local generation
pipeline; its answer quality will usually be worse than the deterministic
grounded stub.

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are an evidence-aware RAG assistant. "
            "Use only the supplied evidence. "
            "Keep citations exactly as written. "
            "If evidence is insufficient, say so and suggest a follow-up."
        ),
    ),
    (
        "human",
        (
            "Question: {question}\n\n"
            "Internal context:\n{context}\n\n"
            "Wikipedia context:\n{wiki}\n\n"
            "Grounded draft:\n{draft}"
        ),
    ),
])


@lru_cache(maxsize=1)
def get_tiny_generator(
    model_id: str = "sshleifer/tiny-gpt2",
):
    """Load the optional local text-generation pipeline once."""

    import torch
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        pipeline,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id
    )

    # GPT-2 models do not define a padding token by default.
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = 0 if torch.cuda.is_available() else -1

    return pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        device=device,
    )


def summarize_with_tiny(
    prompt_text: str,
    max_new_tokens: int = 100,
) -> str:
    """Generate an optional local completion."""

    generator = get_tiny_generator()

    output = generator(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id,
    )

    return output[0]["generated_text"].strip()

In [ ]:
def answer_question(
    question: str,
    use_tiny_model: bool = False,
    wikipedia_k: int = 2,
) -> dict[str, Any]:
    """Plan, retrieve, assess evidence, and return a cited answer."""

    question = question.strip()
    plan_result = plan(question)

    kb_documents: list[Document] = []
    wiki_snippets: list[dict[str, str]] = []
    wiki_error: str | None = None
    used_wiki_fallback = False

    if plan_result["action"] == "none":
        return {
            "plan": plan_result,
            "kb_sources": [],
            "wiki_sources": [],
            "wiki_error": None,
            "used_wiki_fallback": False,
            "evidence_thin": True,
            "answer": (
                "No question was supplied. Please enter a specific "
                "question about the knowledge base or an external topic."
            ),
        }

    # The planner first chooses the primary information source.
    if plan_result["action"] == "kb":
        kb_documents = rerank_retrieved_documents(
            question,
            retriever.invoke(question),
        )

    evidence_thin = kb_evidence_is_thin(
        question,
        kb_documents,
    )

    # Wikipedia is used when:
    # - it is the primary planned action; or
    # - internal evidence is too weak and fallback is allowed.
    should_use_wikipedia = (
        plan_result["action"] == "wiki"
        or (
            evidence_thin
            and plan_result["wiki_fallback_if_thin"]
        )
    )

    if should_use_wikipedia:
        wiki_snippets, wiki_error = wiki_search(
            question,
            k=wikipedia_k,
        )
        used_wiki_fallback = (
            plan_result["action"] == "kb"
        )

    # Recalculate thinness using all available evidence.
    evidence_thin = (
        not kb_documents
        and not wiki_snippets
    ) or (
        evidence_thin
        and not wiki_snippets
    )

    context_text = "\n".join(
        (
            f"[{document.metadata['source']}] "
            f"{document.page_content}"
        )
        for document in kb_documents
    ) or "No internal KB evidence."

    wiki_text = "\n".join(
        (
            f"[{snippet['source_id']}] "
            f"{snippet['summary']}"
        )
        for snippet in wiki_snippets
    ) or "No Wikipedia evidence."

    grounded_draft = build_grounded_draft(
        question=question,
        kb_documents=kb_documents,
        wiki_snippets=wiki_snippets,
        evidence_thin=evidence_thin,
        wiki_error=wiki_error,
    )

    messages = answer_prompt.format_messages(
        question=question,
        context=context_text,
        wiki=wiki_text,
        draft=grounded_draft,
    )

    if use_tiny_model:
        # The optional model receives a text representation of the messages.
        prompt_text = "\n\n".join(
            str(message.content)
            for message in messages
        )
        final_answer = summarize_with_tiny(
            prompt_text
        )

        # A tiny causal model may omit citations. Preserve grounding by
        # falling back to the deterministic draft when that happens.
        if not re.search(
            r"\[(?:kb|wiki):[^\]]+\]",
            final_answer,
        ):
            final_answer = grounded_draft
    else:
        # FakeListChatModel satisfies the stub-LLM requirement while keeping
        # the result deterministic and evidence-backed.
        stub_model = FakeListChatModel(
            responses=[grounded_draft]
        )
        final_answer = stub_model.invoke(
            messages
        ).content

    return {
        "plan": plan_result,
        "kb_sources": [
            document.metadata["source"]
            for document in kb_documents
        ],
        "wiki_sources": [
            snippet["source_id"]
            for snippet in wiki_snippets
        ],
        "wiki_error": wiki_error,
        "used_wiki_fallback": used_wiki_fallback,
        "evidence_thin": evidence_thin,
        "answer": final_answer,
    }

# Exercise 5 — Quick check

The three questions represent:

1. an internal KB topic;
2. a general external fact;
3. a mixed or ambiguous question involving internal policy and Wikipedia.

Stub mode is used so the test remains fast and deterministic.

In [ ]:
test_questions = [
    (
        "KB-covered",
        "How should an agentic system ground and cite its answers?",
    ),
    (
        "External",
        "Who created the Python programming language?",
    ),
    (
        "Ambiguous",
        (
            "How should an agent use Wikipedia when internal "
            "evidence is incomplete?"
        ),
    ),
]

test_results = []

for category, question in test_questions:
    result = answer_question(
        question,
        use_tiny_model=False,
    )

    test_results.append({
        "category": category,
        "question": question,
        "action": result["plan"]["action"],
        "kb_sources": ", ".join(result["kb_sources"]) or "none",
        "wiki_sources": ", ".join(result["wiki_sources"]) or "none",
        "thin": result["evidence_thin"],
        "answer": result["answer"],
    })

    print("=" * 90)
    print("CATEGORY:", category)
    print("QUESTION:", question)
    print("PLAN:", result["plan"])
    print("KB SOURCES:", result["kb_sources"])
    print("WIKIPEDIA SOURCES:", result["wiki_sources"])
    print("WIKIPEDIA ERROR:", result["wiki_error"])
    print("ANSWER:", result["answer"])

display(pd.DataFrame(test_results))

In [ ]:
# Lightweight checks confirm the required behavior without assuming exact
# FAISS document ordering or live Wikipedia availability.

assert plan(
    "How should an agent ground its answer?"
)["action"] == "kb"

assert plan(
    "Who created Python?"
)["action"] == "wiki"

assert len(
    retriever.invoke("retrieval citations")
) == 3

for result in test_results:
    answer = result["answer"]

    # Every evidence-backed answer should contain a source marker.
    # When Wikipedia is unreachable and no evidence exists, the answer must
    # instead state that evidence is insufficient.
    assert (
        re.search(
            r"\[(?:kb|wiki):[^\]]+\]",
            answer,
        )
        or "not provide enough evidence" in answer.lower()
        or "did not contain a clear statement" in answer.lower()
    )

print("Quick checks passed.")

# Optional experiment — Tiny Hugging Face generation

This cell is disabled by default because it downloads a model.

Change the flag to `True` to compare the local generator with the grounded
stub. The answer function automatically falls back to the grounded draft if
the tiny model fails to preserve citations.

In [ ]:
RUN_OPTIONAL_TINY_MODEL = False

if RUN_OPTIONAL_TINY_MODEL:
    tiny_result = answer_question(
        "How should RAG systems handle weak evidence?",
        use_tiny_model=True,
    )

    print("Plan:", tiny_result["plan"])
    print("Sources:", {
        "kb": tiny_result["kb_sources"],
        "wiki": tiny_result["wiki_sources"],
    })
    print("Answer:", tiny_result["answer"])
else:
    print(
        "Set RUN_OPTIONAL_TINY_MODEL=True "
        "to download and test sshleifer/tiny-gpt2."
    )

# Observations

## Retrieval

FAISS provides the expected vector-store and retriever interfaces. Because
the exercise uses fake embeddings, exact semantic relevance is not
guaranteed.

## Planning

A rule-based planner is easy to inspect and debug. It chooses the internal
KB for known topics and Wikipedia for everything else.

## Agent-like fallback

The answer function does not blindly trust the initial plan. It checks the
retrieved evidence and can add a Wikipedia lookup when internal evidence is
weak.

## Grounding

Source identifiers are carried from the original documents or Wikipedia
results into the final answer. The answer function never invents a source
identifier.

## Missing evidence

Wikipedia or network errors become part of the result metadata. The system
returns a transparent limitation statement instead of crashing or guessing.

## Model quality

The stub is intentionally deterministic. `sshleifer/tiny-gpt2` demonstrates
local generation, but it is not an instruction-tuned production RAG model.

# Possible improvements

1. Replace fake embeddings with a local sentence-transformer.
2. Add similarity scores and a calibrated relevance threshold.
3. Use metadata filters for document type, date, or department.
4. Add a reranker after FAISS retrieval.
5. Cache Wikipedia results on disk.
6. Add an authoritative-source tool for sensitive topics.
7. Replace the rule planner with a structured-output local model.
8. Evaluate citation correctness over a test dataset.

# Deliverables checklist

- [x] 5–8 internal `Document` objects
- [x] Source metadata on every document
- [x] `FakeEmbeddings`
- [x] FAISS vector store
- [x] Retriever returning top 3
- [x] Wikipedia helper without an API key
- [x] Short title/snippet output
- [x] Rule-based planner
- [x] KB preference for known topics
- [x] Wikipedia fallback
- [x] Stub chat model by default
- [x] Optional tiny Hugging Face pipeline
- [x] Inline `[kb:...]` and `[wiki:...]` citations
- [x] Thin-evidence handling
- [x] Three sample questions
- [x] Plan, sources, and answer printed
- [x] Thorough code comments and Markdown documentation

# Conclusion

This notebook demonstrates a transparent agentic RAG loop:

```text
plan
  → retrieve
  → inspect evidence
  → optionally use an external tool
  → answer with traceable citations
```

The most important behavior is not tool usage itself. It is the decision to
use the right source, verify that evidence exists, and avoid unsupported
claims when retrieval is weak.

# References

- LangChain vector-store integrations:
  https://docs.langchain.com/oss/python/integrations/vectorstores
- LangChain semantic-search tutorial:
  https://docs.langchain.com/oss/python/langchain/knowledge-base
- WikipediaAPIWrapper reference:
  https://python.langchain.com/api_reference/community/utilities/
- FAISS:
  https://github.com/facebookresearch/faiss